<h2>1. K-Fold walidacja krzyżowa</h2>
<p>Sprawdzam konfiguracje różnych wartości hiperparametrów trenując model kilkukrotnie</p>
<p>Tutaj będe porównywał wyniki, bazując na metryce AUC ROC dla każdej konfiguracji hiperparametrów</p>
<p>1. Pierwsze modele posłużą do wyboru optymalnego współczynnika uczenia - <code>learning_rate</code></p>
<p>2. Kolejne modele do wyboru optymalnego parametru regularyzacji L2 - <code>weight_decay</code></p>
<p>3. Ostatnie modele do wyboru optymalnego rozmiaru batch'a - <code>batch_size</code></p>

<h3>1.1. Import biblioteki <code>pandas</code></h3>

In [66]:
import pandas as pd
import numpy as np

<h3>1.2. Definicja liczby foldów i funkcji pomocniczych</h3>

In [67]:
FOLDS = 5


def k_folds():
    return range(1, FOLDS + 1)


def auc_metrics_by_version(version):
    auc = pd.DataFrame()
    for fold in k_folds():
        fold_auc = pd.read_csv(
            f"kfold_results/{version}/results_DENSENET_121_f{fold}_{version}/auc_metrics_DENSENET_121_f{fold}_{version}.csv"
        )
        if fold > 1:
            fold_auc.drop(columns=["class_name"], inplace=True)
        fold_auc.rename(columns={"AUC": f"AUC_{fold}"}, inplace=True)
        auc = pd.concat([auc, fold_auc], axis=1)
    return auc


def show_auc_metrics(auc_metrics, version):
    avg_auc_by_fold = []
    for fold in k_folds():
        print(f"Średnia dla fold[{fold}] = {auc_metrics[f'AUC_{fold}'].mean():.3f}")
        avg_auc_by_fold.append(auc_metrics[f'AUC_{fold}'].mean())
    print(f"Średnia dla wersji {version} = {np.mean(avg_auc_by_fold):.3f}")


def best_epochs_by_version(version):
    def best_epoch(df):
        return int(df.loc[df["val_loss"].idxmin(), "epoch"])

    best_epochs = []
    for folds in k_folds():
        train_log_df = pd.read_csv(
            f"kfold_results/{version}/results_DENSENET_121_f{folds}_{version}/train_log_DENSENET_121_f{folds}_{version}.log"
        )
        best_epochs.append(best_epoch(train_log_df))
    assert len(best_epochs) == FOLDS
    return pd.Series(best_epochs, dtype=float)


<h2>2. Model V1</h2>
<ul>
<li><code>learning_rate = 0.01</code></li>
<li><code>weight_decay = 1e-4</code></li>
<li><code>batch_size = 16</code></li>
</ul>

<h3>2.1 Metryki AUC dla foldów</h3>

In [68]:
v1_auc = auc_metrics_by_version("v1")
v1_auc


,class_name,AUC_1,AUC_2,AUC_3,AUC_4,AUC_5
0,Atelectasis,0.815135,0.809060,0.802074,0.802300,0.798470
1,Cardiomegaly,0.897403,0.892389,0.908685,0.893679,0.893893
2,Effusion,0.881382,0.875186,0.874543,0.881741,0.878426
3,Infiltration,0.708835,0.709344,0.708767,0.707148,0.706047
4,Mass,0.835818,0.828930,0.848420,0.829774,0.829258
5,Nodule,0.746941,0.758639,0.773740,0.753883,0.770049
6,Pneumonia,0.736753,0.771264,0.752947,0.728226,0.747676
7,Pneumothorax,0.864562,0.881927,0.856539,0.866068,0.861568
8,Consolidation,0.796021,0.812106,0.788051,0.768403,0.802354
9,Edema,0.899608,0.892398,0.895028,0.896399,0.889776


<h3>2.2 Średnia AUC</h3>

In [69]:
show_auc_metrics(v1_auc, "v1")

Średnia dla fold[1] = 0.825
Średnia dla fold[2] = 0.827
Średnia dla fold[3] = 0.829
Średnia dla fold[4] = 0.822
Średnia dla fold[5] = 0.825
Średnia dla wersji v1 = 0.826


<h2>3. Model V2</h2>
<ul>
<li><code>learning_rate = 0.003</code></li>
<li><code>weight_decay = 1e-4</code></li>
<li><code>batch_size = 16</code></li>
</ul>

<h3>3.1 Metryki AUC dla foldów</h3>

In [70]:
v2_auc = auc_metrics_by_version("v2")
v2_auc

,class_name,AUC_1,AUC_2,AUC_3,AUC_4,AUC_5
0,Atelectasis,0.810056,0.805656,0.796989,0.787194,0.795196
1,Cardiomegaly,0.892162,0.871880,0.904600,0.888209,0.885183
2,Effusion,0.878146,0.872054,0.867418,0.881811,0.876367
3,Infiltration,0.711691,0.705638,0.701675,0.709020,0.703489
4,Mass,0.811307,0.815524,0.830300,0.817362,0.825262
5,Nodule,0.730717,0.739813,0.778717,0.736715,0.757074
6,Pneumonia,0.749665,0.763651,0.721422,0.744938,0.761955
7,Pneumothorax,0.858829,0.881278,0.860197,0.861361,0.864513
8,Consolidation,0.797469,0.803526,0.786201,0.777442,0.799036
9,Edema,0.895690,0.886086,0.889368,0.893876,0.896145


<h3>3.2 Średnia AUC</h3>

In [71]:
show_auc_metrics(v2_auc, "v2")

Średnia dla fold[1] = 0.814
Średnia dla fold[2] = 0.815
Średnia dla fold[3] = 0.820
Średnia dla fold[4] = 0.815
Średnia dla fold[5] = 0.822
Średnia dla wersji v2 = 0.817


<h2>4. Model V3</h2>
<ul>
<li><code>learning_rate = 0.001</code></li>
<li><code>weight_decay = 1e-4</code></li>
<li><code>batch_size = 16</code></li>
</ul>

<h3>4.1 Metryki AUC dla foldów</h3>

In [72]:
v3_auc = auc_metrics_by_version("v3")
v3_auc

,class_name,AUC_1,AUC_2,AUC_3,AUC_4,AUC_5
0,Atelectasis,0.802730,0.802253,0.789717,0.789296,0.783506
1,Cardiomegaly,0.892086,0.869660,0.889170,0.884551,0.863729
2,Effusion,0.871319,0.868783,0.865443,0.876569,0.871293
3,Infiltration,0.711238,0.705206,0.701448,0.706731,0.703074
4,Mass,0.810046,0.789405,0.818650,0.799926,0.801672
5,Nodule,0.718687,0.719600,0.754040,0.734285,0.742877
6,Pneumonia,0.746028,0.759038,0.742028,0.724843,0.744489
7,Pneumothorax,0.851348,0.870950,0.869010,0.865898,0.862591
8,Consolidation,0.800972,0.806072,0.781367,0.769266,0.795255
9,Edema,0.893594,0.886266,0.889435,0.897171,0.891168


<h3>4.2 Średnia AUC</h3>

In [73]:
show_auc_metrics(v3_auc, "v3")

Średnia dla fold[1] = 0.809
Średnia dla fold[2] = 0.807
Średnia dla fold[3] = 0.815
Średnia dla fold[4] = 0.800
Średnia dla fold[5] = 0.807
Średnia dla wersji v3 = 0.808


<h2>5. Model V4</h2>
<ul>
<li><code>learning_rate = 0.03</code></li>
<li><code>weight_decay = 1e-4</code></li>
<li><code>batch_size = 16</code></li>
</ul>

<h3>5.1 Metryki AUC dla foldów</h3>

In [74]:
v4_auc = auc_metrics_by_version("v4")
v4_auc

,class_name,AUC_1,AUC_2,AUC_3,AUC_4,AUC_5
0,Atelectasis,0.815016,0.810250,0.802228,0.794707,0.801669
1,Cardiomegaly,0.901273,0.883811,0.909124,0.873563,0.897736
2,Effusion,0.875717,0.874440,0.870145,0.877847,0.871291
3,Infiltration,0.707658,0.705973,0.703517,0.697030,0.696951
4,Mass,0.830956,0.821118,0.833948,0.829903,0.836147
5,Nodule,0.741731,0.725411,0.774635,0.751610,0.760288
6,Pneumonia,0.753903,0.755722,0.732370,0.740038,0.767899
7,Pneumothorax,0.863787,0.875853,0.862037,0.868108,0.865458
8,Consolidation,0.811655,0.807375,0.794469,0.779704,0.797017
9,Edema,0.900880,0.894768,0.884226,0.894558,0.888953


<h3>5.2 Średnia AUC</h3>

In [75]:
show_auc_metrics(v4_auc, "v4")

Średnia dla fold[1] = 0.820
Średnia dla fold[2] = 0.817
Średnia dla fold[3] = 0.824
Średnia dla fold[4] = 0.808
Średnia dla fold[5] = 0.819
Średnia dla wersji v4 = 0.818


In [76]:
# Średnia dla wersji v1 = 0.826
# Średnia dla wersji v2 = 0.817
# Średnia dla wersji v3 = 0.808
# Średnia dla wersji v4 = 0.818

<h2>Najwyższą średnią AUC ma model V1, wybieram <code>learning_rate = 0.01</code> i przechodzę do wyboru optymalnego parametru <code>weight_decay</code>

<h2>6. Model V5</h2>
<ul>
<li><code><strong>learning_rate = 0.01</strong></code></li>
<li><code>weight_decay = 5e-4</code></li>
<li><code>batch_size = 16</code></li>
</ul>

<h3>6.1 Metryki AUC dla foldów</h3>

In [77]:
v5_auc = auc_metrics_by_version("v5")
v5_auc

,class_name,AUC_1,AUC_2,AUC_3,AUC_4,AUC_5
0,Atelectasis,0.810593,0.807913,0.795734,0.791708,0.799889
1,Cardiomegaly,0.893100,0.892671,0.893605,0.876577,0.882691
2,Effusion,0.876765,0.871305,0.867716,0.880449,0.877870
3,Infiltration,0.705814,0.701200,0.707295,0.704174,0.697833
4,Mass,0.816187,0.830573,0.832476,0.801923,0.828809
5,Nodule,0.737577,0.743796,0.771965,0.744312,0.753396
6,Pneumonia,0.729245,0.761951,0.739840,0.729981,0.748610
7,Pneumothorax,0.854520,0.875485,0.872187,0.864291,0.856252
8,Consolidation,0.803500,0.812811,0.793543,0.763530,0.793218
9,Edema,0.885144,0.888636,0.892871,0.896607,0.882781


<h3>6.2 Średnia AUC</h3>

In [78]:
show_auc_metrics(v5_auc, "v5")

Średnia dla fold[1] = 0.811
Średnia dla fold[2] = 0.815
Średnia dla fold[3] = 0.818
Średnia dla fold[4] = 0.803
Średnia dla fold[5] = 0.805
Średnia dla wersji v5 = 0.810


<h2>7. Model V6</h2>
<ul>
<li><code><strong>learning_rate = 0.01</strong></code></li>
<li><code>weight_decay = 0</code></li>
<li><code>batch_size = 16</code></li>
</ul>

<h3>7.1 Metryki AUC dla foldów</h3>

In [79]:
v6_auc = auc_metrics_by_version("v6")
v6_auc

,class_name,AUC_1,AUC_2,AUC_3,AUC_4,AUC_5
0,Atelectasis,0.816656,0.811326,0.810510,0.806829,0.796768
1,Cardiomegaly,0.895999,0.879544,0.903663,0.889711,0.892774
2,Effusion,0.880664,0.877461,0.870899,0.888050,0.880455
3,Infiltration,0.706828,0.714674,0.703559,0.707083,0.710548
4,Mass,0.836175,0.819009,0.842503,0.827014,0.834284
5,Nodule,0.757392,0.743307,0.785602,0.747687,0.749238
6,Pneumonia,0.752168,0.767122,0.750198,0.754908,0.760453
7,Pneumothorax,0.862971,0.878914,0.869229,0.874070,0.866014
8,Consolidation,0.802141,0.811397,0.795051,0.783945,0.803105
9,Edema,0.898780,0.888523,0.891221,0.900987,0.890579


<h3>7.2 Średnia AUC</h3>

In [80]:
show_auc_metrics(v6_auc, "v6")

Średnia dla fold[1] = 0.824
Średnia dla fold[2] = 0.822
Średnia dla fold[3] = 0.831
Średnia dla fold[4] = 0.826
Średnia dla fold[5] = 0.825
Średnia dla wersji v6 = 0.825


In [81]:
# Średnia dla wersji v1 = 0.826
# Średnia dla wersji v5 = 0.810
# Średnia dla wersji v6 = 0.825

<h2>Najwyższą średnią AUC ma ponownie model V1, wybieram <code>weight_decay = 1e-4</code> i przechodzę do wyboru optymalnego parametru <code>batch_size</code>

<h2>8. Model V7</h2>
<ul>
<li><code><strong>learning_rate = 0.08</strong></code> (8x większy, ponieważ dostosowany do rozmiaru batch'a)</li>
<li><code><strong>weight_decay = 1e-4</strong></code></li>
<li><code>batch_size = 128</code></li>
</ul>

<h3>8.1 Metryki AUC dla foldów</h3>

In [82]:
v7_auc = auc_metrics_by_version("v7")
v7_auc

,class_name,AUC_1,AUC_2,AUC_3,AUC_4,AUC_5
0,Atelectasis,0.819413,0.813292,0.795488,0.790435,0.798090
1,Cardiomegaly,0.895899,0.868542,0.908158,0.893502,0.882660
2,Effusion,0.879909,0.875652,0.867000,0.884183,0.879770
3,Infiltration,0.718370,0.704920,0.709502,0.703257,0.703694
4,Mass,0.840554,0.821308,0.842018,0.814685,0.822253
5,Nodule,0.746512,0.752312,0.766361,0.751383,0.764656
6,Pneumonia,0.761664,0.775525,0.734072,0.729923,0.759232
7,Pneumothorax,0.857560,0.875321,0.868639,0.852739,0.872171
8,Consolidation,0.815367,0.809775,0.786949,0.770388,0.801810
9,Edema,0.894977,0.889029,0.885055,0.903197,0.894554


<h3>8.2 Średnia AUC</h3>

In [83]:
show_auc_metrics(v7_auc, "v7")

Średnia dla fold[1] = 0.828
Średnia dla fold[2] = 0.821
Średnia dla fold[3] = 0.825
Średnia dla fold[4] = 0.809
Średnia dla fold[5] = 0.821
Średnia dla wersji v7 = 0.821


In [84]:
# Średnia dla wersji v1 = 0.826
# Średnia dla wersji v7 = 0.821

<h2>Zmiana rozmiaru batcha nie wpłynęła na wyniki, najwyższą średnią AUC ma model V1 wytrenowany podstawowych wartościach parametrów:
<ul>
<li><code>learning_rate = 0.01</code></li>
<li><code>weight_decay = 1e-4</code></li>
<li><code>batch_size = 16</code></li>
</ul>

<h2>9. Wybór ilości epok do treningu modelu na cały zbiorze treningowym</h2>

<h3>9.1. Wyświetlam ilość epok treningu ile zajęło wytrenowanie najlepszego modelu w wersji V1 dla każdego fold'a</h3>

In [85]:
best_epoch = best_epochs_by_version("v1")
for fold in k_folds():
    print(f"Fold {fold} ilość epok = {best_epoch[fold - 1]}")


Fold 1 ilość epok = 6.0
Fold 2 ilość epok = 6.0
Fold 3 ilość epok = 4.0
Fold 4 ilość epok = 4.0
Fold 5 ilość epok = 4.0


<h3>9.2. Średnia ilość epok dla modelu V1</h3>

In [86]:
print(f"Średnia ilość epok dla najlepszej wersji modelu V1: {best_epoch.mean():.2f}")

Średnia ilość epok dla najlepszej wersji modelu V1: 4.80


<h1>Wybieram 5 jako liczbę epok treningu modelu docelowego</h1>